# Aula 5 — Fechamento e apresentação

**Projeto final · Laboratório de Controle Automático · Ifes Guarapari · 10/12 · 1h40**

**Objetivo da aula:** juntar os resultados das quatro aulas num **painel** e apresentá-lo. Não há
slides: a apresentação **é** este notebook.

**Papéis na bancada (troquem a cada aula):** 🔧 **operador(a)** — mexe no hardware e grava o
firmware · 💻 **programador(a)** — roda e edita o notebook · ✍️ **relator(a)** — escreve as
respostas ❓ e confere os números. Com 2 pessoas, o relator acumula com o programador.

**Legenda:** ⏱️ tempo sugerido · 📘 teoria (leia antes de fazer) · 🔧 bancada · 💻 notebook ·
✏️ complete o código · ❓ responda · ✅ checkpoint — **chame o professor**.

| ⏱️ | Parte | O que se faz |
|---|---|---|
| 0–20 min | 1 | Gerar o painel de resultados |
| 20–30 min | 2 | Tabela das não linearidades e conclusão |
| 30–100 min | 3 | Apresentações (≈ 10 min + 3 min de perguntas por bancada) |

In [ ]:
# ⚙️ CONFIGURAÇÃO — rode esta célula sempre que abrir o notebook
import numpy as np
import matplotlib.pyplot as plt
import control as ct
from scipy.optimize import minimize
from ferramentas_projeto import *

PORTA = "COM5"     # ✏️ porta do Arduino (Arduino IDE > Ferramentas > Porta). Linux: "/dev/ttyUSB0" ou "/dev/ttyACM0"
BANCADA = "1"      # ✏️ número da bancada

s = ct.tf("s")
Ts = 0.02          # período de amostragem do firmware [s]
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})
P = carregar_parametros()          # resultados das aulas anteriores (dados/parametros_bancada.json)
print("bancada", BANCADA, "| parâmetros já salvos:", list(P))

## Parte 1 — Painel de resultados · ⏱️ 20 min

As células abaixo leem **todos** os arquivos de `dados/` e os parâmetros salvos. Se alguma falhar,
é porque falta um ensaio de uma aula anterior — chame o professor.

In [ ]:
c = P["CFG_FINAL"]
print(f"══════ BANCADA {BANCADA} — servo de posição com motor CC ══════\n")
print("1) MODELO IDENTIFICADO (Aulas 1–2)")
print(f"   encoder: CPG = {P['CPG']:.4f} contas/grau (resolução {1/P['CPG']:.2f}°)")
print(f"   velocidade (PRBS): K = {P['K']:.4f}, tau = {P['tau']:.3f} s, L = {1e3*P['L']:.0f} ms, u0 = {P['u0']:.0f} PWM")
print(f"   posição: G_p(s) = {P['Km']:.2f} / [s ({P['tau']:.3f} s + 1)]  graus/PWM")
print(f"   zona morta: +{P['Fc_pos']:.0f} / −{P['Fc_neg']:.0f} PWM   ·   partida: +{P['Fs_pos']:.0f} / −{P['Fs_neg']:.0f} PWM")
print("\n2) CONTROLADORES (Aulas 2–3)")
i1, zn = P["iter1"], P["ZN"]
print(f"   iteração 1 (LGR, PI)      : kp = {i1['kp']:.2f}  ki = {i1['ki']:.2f}  kd = 0")
print(f"   Ziegler-Nichols (relé)    : kp = {zn['kp']:.2f}  ki = {zn['ki']:.1f}  kd = {zn['kd']:.3f}")
print(f"   FINAL (LGR, PID + compens.): kp = {c['KP']}  ki = {c['KI']}  kd = {c['KD']}  N = {c['N']}  ZM = {c['ZM']}  EM = {c['EM']}")
print("\n3) ESPECIFICAÇÕES NA BANCADA — degrau de 90°, pior de 3 repetições (Aula 4)")
veredito(P["pior_nominal"])
print(("✅" if P["dMp_aw1"] <= 5 else "❌") +
      f" 4. ΔMp com anti-windup = {P['dMp_aw1']:+.1f} p.p. ≤ 5 p.p.  (sem anti-windup: {P['dMp_aw0']:+.1f} p.p.)")

In [ ]:
it1 = carregar_csv("dados/mf_iter1.csv")
fin = carregar_csv("dados/mf_nominal_1.csv")
sat1, sat0 = carregar_csv("dados/mf_sat_aw1.csv"), carregar_csv("dados/mf_sat_aw0.csv")
ma = carregar_csv("dados/ma_p100.csv")
sim_ma = simular(P["GEMEO"], MODO=1, PWM_MA=100)

fig, ax = plt.subplots(2, 2, figsize=(13, 7.5))
ax[0, 0].plot(ma["t"], ma["angle"], color="0.6", lw=3, label="bancada")
ax[0, 0].plot(sim_ma["t"], sim_ma["angle"], "b", label="gêmeo digital")
ax[0, 0].set_title("(a) O modelo reproduz a bancada (malha aberta, PWM 100)")
ax[0, 1].step(fin["t"], fin["r"], "k--", where="post", lw=1, label="referência")
ax[0, 1].plot(it1["t"], it1["angle"], "tab:red", label="iteração 1 (só PI)")
ax[0, 1].plot(fin["t"], fin["angle"], "tab:blue", label="final (PID + compensação)")
ax[0, 1].set_title("(b) O que o atrito fazia × o controlador final")
ax[1, 0].plot(sat0["t"], sat0["angle"], "tab:red", label="sem anti-windup")
ax[1, 0].plot(sat1["t"], sat1["angle"], "tab:blue", label="com anti-windup")
ax[1, 0].axhline(90, color="k", ls="--", lw=0.8); ax[1, 0].set_xlim(1.5, 7); ax[1, 0].set_ylim(-5, 110)
ax[1, 0].set_title("(c) Especificação 4: PWM limitado")
ax[1, 1].plot(sat0["t"], sat0["u_pid"], "tab:red", ls=":", label="u pedido — sem AW")
ax[1, 1].plot(sat1["t"], sat1["u_pid"], "tab:blue", ls=":", label="u pedido — com AW")
ax[1, 1].plot(sat1["t"], sat1["pwm"], "k", lw=1, label="PWM aplicado")
ax[1, 1].set_xlim(1.9, 4.5); ax[1, 1].set_ylim(-60, 250)
ax[1, 1].set_title("(d) O integrador durante a saturação")
for a_ in ax.flat:
    a_.legend(fontsize=8); a_.set_xlabel("tempo (s)")
ax[0, 0].set_ylabel("ângulo (graus)"); ax[1, 0].set_ylabel("ângulo (graus)"); ax[1, 1].set_ylabel("PWM")
fig.suptitle(f"Bancada {BANCADA}"); fig.tight_layout(); plt.show()

## Parte 2 — Não linearidades e conclusão · ⏱️ 10 min

❓ **2.1** Complete a tabela: onde cada não linearidade apareceu **nos seus dados** (cite a aula e
o gráfico ou número) e como foi tratada.

| Não linearidade | Onde apareceu | Como foi tratada |
|---|---|---|
| Saturação do PWM | | |
| Zona morta (atrito de Coulomb) | | |
| Atrito estático | | |
| Resolução do encoder | | |
| Atraso | | |

✍️ **Resposta da bancada:** *(escreva aqui)*

❓ **2.2** Conclusão da bancada, em três frases: o que funcionou, o que o modelo linear não previa,
e o que vocês fariam diferente com mais tempo.

✍️ **Resposta da bancada:** *(escreva aqui)*

## Parte 3 — Apresentação · ⏱️ ≈ 10 min + 3 min de perguntas por bancada

Apresentem **este notebook**, rolando a tela. Dividam a fala entre todos da bancada:

1. **O modelo** (2 min) — painel item 1 e figura (a): de onde saiu cada número.
2. **A iteração 1** (2 min) — figura (b) em vermelho: por que o projeto "certo no papel" falhou.
3. **O controlador final** (3 min) — o que mudou (polos mais rápidos, PD, compensação) e o veredito
   das especificações 1–3.
4. **O anti-windup** (2 min) — figuras (c) e (d): onde se vê o integrador carregando.
5. **As não linearidades e a conclusão** (1 min) — a tabela da Parte 2.

Na arguição, o professor vai perguntar sobre os **seus** gráficos e números, a qualquer integrante.

**Entrega:** a pasta do projeto da bancada com os 5 notebooks **salvos com as saídas** e a pasta
`dados/` completa, no local indicado pelo professor.